#Initialization

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
dbutils.widgets.text("load_type", "incremental")
load_type = dbutils.widgets.get("load_type")

#Silver Transformation

In [0]:
def silver_transformstion(df):
    df = df.filter(F.col("order_qty").isNotNull())
    
    df = df.withColumn(
        "customer_id",
        F.when(F.col("customer_id").rlike("^[0-9]+$"), F.col("customer_id"))
         .otherwise("999999")
         .cast("string")
    )
    
    df = df.withColumn(
        "order_placement_date",
        F.regexp_replace(F.col("order_placement_date"), r"^[A-Za-z]+,\s*", "")
    )
    
    df = df.withColumn(
        "order_placement_date",
        F.coalesce(
            F.try_to_date("order_placement_date", "yyyy/MM/dd"),
            F.try_to_date("order_placement_date", "dd-MM-yyyy"),
            F.try_to_date("order_placement_date", "dd/MM/yyyy"),
            F.try_to_date("order_placement_date", "MMMM dd, yyyy"),
        )
    )
    
    df= df.dropDuplicates(["order_id", "order_placement_date", "customer_id", "product_id",     "order_qty"])
    
    df = df.withColumn('product_id', F.col('product_id').cast('string'))

    return df

#Join With product table

In [0]:
def join_with_products(df):
    df_products = spark.table("pcat.silver.products")
    df_joined = df.join(df_products, on="product_id", how="inner").select(df["*"], df_products["product_code"])

    return df_joined

#Merge with silver layer

In [0]:
def merge_silver(df):
    if not (spark.catalog.tableExists("pcat.silver.orders")):
        df.write.format("delta").option(
            "delta.enableChangeDataFeed", "true"
        ).option("mergeSchema", "true").mode("overwrite").saveAsTable("pcat.silver.orders")
    else:
        silver_delta = DeltaTable.forName(spark, "pcat.silver.orders")
        silver_delta.alias("silver").merge(
            source=df.alias("bronze"),
            condition="silver.order_placement_date = bronze.order_placement_date AND silver.order_id = bronze.order_id AND silver.product_code = bronze.product_code AND silver.customer_id = bronze.customer_id").whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

# Staging table to process just the arrived incremenal data

In [0]:
def stage_silver_table(df):
    df.write \
    .format("delta") \
    .option("delta.enableChangeDataFeed", "true") \
    .mode("overwrite") \
    .saveAsTable("pcat.silver.staging_orders")

#Full Load

In [0]:
def full_load():
    df = spark.table("pcat.bronze.orders")
    try:
        df = silver_transformstion(df)
        df = join_with_products(df)
        merge_silver(df)
        print("Full load completed successfully.")
    except Exception as e:
        print(f"Incremental load failed: {e}")
        raise

#Incremental Load

In [0]:
def incremental_load():
    df = spark.table("pcat.bronze.staging_orders")

    try:
        df = silver_transformstion(df)
        df = join_with_products(df)
        merge_silver(df)
        stage_silver_table(df)

        spark.sql("TRUNCATE TABLE pcat.bronze.staging_orders")
        print("Incremental load completed successfully.")
    except Exception as e:
        print(f"Incremental load failed: {e}")
        raise

##Check silver orders table

In [0]:
df_silver = spark.read.table("pcat.silver.orders")
display(df_silver.orderBy(F.desc("read_timestamp")).limit(15))

#Orchestrate the loading process

In [0]:
if load_type == "full":
    print("performing silver full load ----")
    full_load()

elif load_type == "incremental":
    print("performing silver incremental load ----")
    incremental_load()